# IPIO: time-to-irAE landmark survival analysis

End-to-end driver for the IPIO survival pipeline: raw EHR data → consolidated
longitudinal data → landmark prediction inputs → Cox models. This pipeline
predicts **time-to-immune-related-adverse-event (irAE)**, using death and
administrative censoring as competing/right-censoring events, with covariates
drawn from labs, somatic tumor genomics, cancer type, and the clinical
covariates GENDER / AGE / pd1pdl1 / ctla4 (checkpoint-inhibitor class). Each
section invokes the underlying script via `!{PYTHON} …`, so one kernel runs
the whole chain.

**Before running:** make sure the active kernel is the `survlatent_ode` conda
env (or activate it inline with `conda run -n survlatent_ode python …`). Edit
the `Paths` cell to point at your data.

## Paths

Defaults point at the cluster filesystem under `/data/gusev/...`. Override
`PROJECT_ROOT`, `DATA`, `INPUTS_DIR`, `OUTPUT_DIR` for local copies.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path("/data/gusev/USERS/jpconnor/code/CAIA")
SURVIVAL_DIR = PROJECT_ROOT / "IPIO" / "survival_analysis"
DATA_PREPROCESSING_DIR = PROJECT_ROOT / "IPIO" / "data_preprocessing"

IPIO_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/IPIO/")
DATA = IPIO_PROJ_PATH / "longitudinal_prediction_data.csv"  # output of step 2 (longitudinal_data_processing.py)
SOMATIC_PATH = Path("/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/clinical_and_genomic_features/complete_somatic_data_df.csv.gz")

INPUTS_DIR = IPIO_PROJ_PATH / "survival_analysis" / "prediction_inputs"
GENOMIC_INPUTS_DIR = INPUTS_DIR / "genomic"
OUTPUT_DIR = IPIO_PROJ_PATH / "survival_analysis" / "local_runs"

os.chdir(PROJECT_ROOT)
INPUTS_DIR.mkdir(parents=True, exist_ok=True)
GENOMIC_INPUTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use the kernel's own interpreter for !python calls -- `!python` would
# otherwise resolve to whatever `python` is on $PATH (often the wrong env).
PYTHON = sys.executable

# BLAS thread caps (mirror the bash scripts so models don't oversubscribe cores).
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# Single endpoint for this project: time-to-irAE, censored on death/administrative censoring.
ENDPOINTS = {"irae": {"duration_col": "t_irae", "event_col": "IRAE"}}

print("python:            ", PYTHON)
print("cwd:               ", os.getcwd())
print("survival_dir:      ", SURVIVAL_DIR)
print("longitudinal_data: ", DATA)
print("somatic_data:      ", SOMATIC_PATH)
print("inputs_dir:        ", INPUTS_DIR)
print("genomic_inputs_dir:", GENOMIC_INPUTS_DIR)
print("output_dir:        ", OUTPUT_DIR)

## 1. Compile the irAE cohort

Runs `data_preprocessing/compile_irae_data.py`, which builds the raw
patient-level irAE cohort table (diagnosis dates, treatment starts, checkpoint
inhibitor class, death/censoring) that `longitudinal_data_processing.py`
consumes downstream. No CLI arguments -- paths are hardcoded inside the
script.

In [ ]:
!{PYTHON} {PROJECT_ROOT}/IPIO/data_preprocessing/compile_irae_data.py

## 2. Build consolidated longitudinal data

Runs `longitudinal_data_processing.py`, which merges the compiled irAE cohort
with longitudinal labs and writes `longitudinal_prediction_data.csv` (`DATA`
above). No CLI arguments.

In [ ]:
!{PYTHON} {DATA_PREPROCESSING_DIR}/longitudinal_data_processing.py

## 3. Sanity-check the consolidated output

Spot-check `DATA` before building landmark inputs: patient count, `IRAE`
event rate, lab-row count, and the date range covered. Column names are
probed defensively since the exact schema is owned by
`longitudinal_data_processing.py`.

In [ ]:
import pandas as pd

df = pd.read_csv(DATA, low_memory=False)
print(f"rows: {len(df):,}")

mrn_col = next((c for c in ("MRN", "mrn", "PATIENT_ID") if c in df.columns), df.columns[0])
n_patients = df[mrn_col].nunique()
print(f"unique patients ({mrn_col}): {n_patients:,}")

if "IRAE" in df.columns:
    # IRAE / t_irae are patient-level outcome columns -- dedupe on patient id
    # before computing the event rate so repeated lab rows don't distort it.
    outcome = df.drop_duplicates(subset=[mrn_col])["IRAE"]
    print(f"IRAE event rate: {outcome.mean():.3f}  ({int(outcome.sum())} / {len(outcome)} patients)")
else:
    print("column 'IRAE' not found -- check longitudinal_data_processing.py output schema")

lab_col = next((c for c in ("collapsed_measurement", "measurement", "LAB_NAME") if c in df.columns), None)
if lab_col is not None:
    print(f"lab rows (non-null {lab_col}): {df[lab_col].notna().sum():,}")
else:
    print("no recognized lab-name column found")

date_col = next((c for c in ("measurement_date", "RESULT_DATE", "date") if c in df.columns), None)
if date_col is not None:
    dates = pd.to_datetime(df[date_col], errors="coerce")
    print(f"date range ({date_col}): {dates.min()} -> {dates.max()}")
else:
    print("no recognized date column found")

## 4. Wipe cached per-landmark inputs

Clears everything `build_prediction_inputs.py` caches under `INPUTS_DIR`,
plus the optional genomic arm outputs, so the next build cells rebuild from
scratch (needed after any change to `DATA`, `SOMATIC_PATH`, or the build
scripts themselves).

In [ ]:
# Wipe cached per-landmark inputs so they're rebuilt from the current data.
for pattern in (
    "aggregated_landmark*.csv",
    "pre_treatment_lab_long_landmark*.csv",
):
    for p in INPUTS_DIR.glob(pattern):
        p.unlink()
        print(f"  removed {p.name}")
for fname in (
    "canonical_labs_train_val.csv",
    "build_manifest.json",
    "split_assignments.csv",
    "landmark_mrn_availability.csv",
    "landmark_attrition.json",
):
    p = INPUTS_DIR / fname
    if p.exists():
        p.unlink()
        print(f"  removed {p.name}")
genomic_dir = INPUTS_DIR / "genomic"
if genomic_dir.exists():
    for pattern in (
        "genomic_aggregated.csv",
        "aggregated_landmark*.csv",
        "pre_sample_lab_long.csv",
        "pre_treatment_lab_long_landmark*.csv",
        "genomic_canonical_labs_train_val.csv",
        "canonical_labs_train_val.csv",
        "genomic_build_manifest.json",
        "build_manifest.json",
    ):
        for p in genomic_dir.glob(pattern):
            p.unlink()
            print(f"  removed genomic/{p.name}")
print("  cache cleared\n")

## 5. Build prediction inputs

Builds the landmark-indexed labs/covariates inputs (train/val/test split,
canonical lab list, per-landmark aggregated feature tables) at landmarks 0
and +90 days.

In [ ]:
!{PYTHON} {DATA_PREPROCESSING_DIR}/build_prediction_inputs.py \
    --data {DATA} \
    --output-dir {INPUTS_DIR} \
    --landmark-days 0 90 \
    --time-unit-days 7 \
    --test-frac 0.20 \
    --val-frac 0.20 \
    --min-patient-coverage 0.20

## 6. Build genomic inputs

COMPASS's own `COMPASS_run_locally.ipynb` doesn't call this script, but somatic
genomics is a first-class covariate set for IPIO, so build the genomic arm
here too. Points `--inputs-dir` at the same `INPUTS_DIR` produced above so the
genomic arm reuses the main train/val/test split rather than re-splitting.

In [ ]:
!{PYTHON} {DATA_PREPROCESSING_DIR}/build_genomic_inputs.py \
    --data {DATA} \
    --somatic-path {SOMATIC_PATH} \
    --inputs-dir {INPUTS_DIR} \
    --time-unit-days 7 \
    --min-patient-coverage 0.20

## 7. Run univariate + multivariate tasks

Runs the standard lab landmark arm at 0d and +90d, plus genomic landmark-0
arms using `prediction_inputs/genomic/`: genomics-only and genomics+labs.
Genomic outputs are written under `local_runs/genomic/` so the lab and genomic
figures can be loaded separately.

In [ ]:
import subprocess
import time

N_FOLDS = 5
FORCE_RERUN = True  # rerun every task and overwrite previous results

# (arm, model, landmark, config_dir, metrics_filename, feature_subset)
# arm="labs" uses INPUTS_DIR and writes to OUTPUT_DIR/{cox,xgboost}/...
# arm="genomics" uses GENOMIC_INPUTS_DIR and writes to OUTPUT_DIR/genomic/{cox,xgboost}/...
# config_dir is the output subfolder; feature_subset controls model candidates.
# "baseline" ignores feature_subset and uses clinical covariates only.
TASKS = [
    ("labs",     "univariate",   0,  "both",               "cox_agg_univariate_nobs_adjusted.csv", "labs"),
    ("labs",     "univariate",   90, "both",               "cox_agg_univariate_nobs_adjusted.csv", "labs"),
    ("labs",     "elastic-net",  0,  "both",               "cox_agg_multivariable_metrics.csv", "labs"),
    ("labs",     "elastic-net",  90, "both",               "cox_agg_multivariable_metrics.csv", "labs"),
    ("labs",     "elastic-net",  0,  "baseline",           "cox_agg_baseline_metrics.csv", "labs"),
    ("labs",     "elastic-net",  90, "baseline",           "cox_agg_baseline_metrics.csv", "labs"),
    ("labs",     "xgboost",      0,  "both",               "landmark_xgboost_metrics.csv", "labs"),
    ("labs",     "xgboost",      90, "both",               "landmark_xgboost_metrics.csv", "labs"),
    ("labs",     "xgboost",      0,  "baseline",           "landmark_xgboost_baseline_metrics.csv", "labs"),
    ("labs",     "xgboost",      90, "baseline",           "landmark_xgboost_baseline_metrics.csv", "labs"),
    ("genomics", "univariate",   0,  "genomics",           "cox_agg_univariate_nobs_adjusted.csv", "genomics"),
    ("genomics", "elastic-net",  0,  "genomics",           "cox_agg_multivariable_metrics.csv", "genomics"),
    ("genomics", "elastic-net",  0,  "genomics_plus_labs", "cox_agg_multivariable_metrics.csv", "all"),
    ("genomics", "elastic-net",  0,  "baseline",           "cox_agg_baseline_metrics.csv", "all"),
    ("genomics", "xgboost",      0,  "genomics",           "landmark_xgboost_metrics.csv", "genomics"),
    ("genomics", "xgboost",      0,  "genomics_plus_labs", "landmark_xgboost_metrics.csv", "all"),
    ("genomics", "xgboost",      0,  "baseline",           "landmark_xgboost_baseline_metrics.csv", "all"),
]


def task_inputs_dir(arm):
    if arm == "labs":
        return INPUTS_DIR
    if arm == "genomics":
        return GENOMIC_INPUTS_DIR
    raise ValueError(f"Unknown arm: {arm}")


def arm_output_root(arm):
    if arm == "labs":
        return OUTPUT_DIR
    if arm == "genomics":
        return OUTPUT_DIR / "genomic"
    raise ValueError(f"Unknown arm: {arm}")


def model_output_dir(model):
    # Univariate + elastic-net share the cox/ output tree; XGBoost gets its own tree.
    return "cox" if model in ("univariate", "elastic-net") else "xgboost"


def task_output_dir(arm, model, landmark, config_dir):
    return arm_output_root(arm) / model_output_dir(model) / f"landmark_{landmark}" / config_dir


def build_command(arm, model, landmark, config_dir, row_output_dir, feature_subset):
    inputs_dir = task_inputs_dir(arm)
    if model == "univariate":
        return [
            PYTHON, str(SURVIVAL_DIR / "univariate_analysis.py"),
            "--inputs-dir", str(inputs_dir),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "irae",
            "--feature-subset", feature_subset,
        ]
    if model == "elastic-net":
        cmd = [
            PYTHON, str(SURVIVAL_DIR / "multivariate_analysis.py"),
            "--model", "elastic-net",
            "--inputs-dir", str(inputs_dir),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "irae",
            "--n-folds", str(N_FOLDS),
        ]
        if config_dir == "baseline":
            cmd.append("--baseline")  # clinical-covariates-only baseline
        else:
            cmd.extend(["--feature-subset", feature_subset])
        return cmd
    if model == "xgboost":
        cmd = [
            PYTHON, str(SURVIVAL_DIR / "multivariate_analysis.py"),
            "--model", "xgboost",
            "--inputs-dir", str(inputs_dir),
            "--output-dir", str(row_output_dir),
            "--landmark-days", str(landmark),
            "--endpoints", "irae",
            "--n-folds", str(N_FOLDS),
        ]
        if config_dir == "baseline":
            cmd.append("--baseline")
        else:
            cmd.extend(["--feature-subset", feature_subset])
        return cmd
    raise ValueError(f"Unknown model: {model}")


summary = []
for arm, model, landmark, config_dir, metrics_filename, feature_subset in TASKS:
    row_output_dir = task_output_dir(arm, model, landmark, config_dir)
    metrics_path = row_output_dir / metrics_filename
    tag = f"{arm:8s}  {model:11s}  landmark_{landmark:<2}  {config_dir:19s}  {feature_subset}"

    if metrics_path.exists() and not FORCE_RERUN:
        print(f"[skip] {tag}  ->  {metrics_path.relative_to(OUTPUT_DIR)} exists")
        summary.append((tag, "skipped", 0.0))
        continue

    row_output_dir.mkdir(parents=True, exist_ok=True)
    cmd = build_command(arm, model, landmark, config_dir, row_output_dir, feature_subset)
    print(f"[run ] {tag}")
    print("       " + " ".join(cmd))
    t0 = time.time()
    rc = subprocess.call(cmd)
    elapsed = time.time() - t0
    status = "ok" if rc == 0 else f"FAILED (rc={rc})"
    print(f"[done] {tag}  -> {status}  ({elapsed/60:.1f} min)\n")
    summary.append((tag, status, elapsed))

print("\n=== run summary ===")
for tag, status, elapsed in summary:
    print(f"  {tag}  {status:>20s}  {elapsed/60:6.1f} min")


## 8. Inspect outputs

Two summaries: (a) the labs-vs-baseline C-index/AUC(t)/Brier comparison
from the elastic-net and XGBoost multivariate tasks (metrics-row schema,
landmark-based -- column names differ between the two model types, normalized
below so they're directly comparable), and (b) a significant-hit count from
the univariate task (per-feature association schema, read separately).


In [ ]:
import pandas as pd

# Elastic-net and XGBoost both write a single metrics row per (arm, landmark, config)
# but under different column names (Cox: test_c_index / test_mean_auc_t /
# test_integrated_brier / n_events_test / landmark_days; XGBoost: c_index /
# mean_auc_t / integrated_brier / n_test_events / landmark_day). Normalize them
# here so both appear in one comparable table. Univariate writes a per-feature
# association table instead, summarized in the next cell.
METRIC_COLUMN_ALIASES = {
    "elastic-net": {
        "landmark": "landmark_days", "n_test": "n_test", "n_test_events": "n_events_test",
        "c_index": "test_c_index", "mean_auc_t": "test_mean_auc_t", "integrated_brier": "test_integrated_brier",
    },
    "xgboost": {
        "landmark": "landmark_day", "n_test": "n_test", "n_test_events": "n_test_events",
        "c_index": "c_index", "mean_auc_t": "mean_auc_t", "integrated_brier": "integrated_brier",
    },
}

rows = []
for arm, model, landmark, config_dir, metrics_filename, feature_subset in TASKS:
    if model not in METRIC_COLUMN_ALIASES:
        continue
    cols = METRIC_COLUMN_ALIASES[model]
    row_output_dir = task_output_dir(arm, model, landmark, config_dir)
    metrics_path = row_output_dir / metrics_filename
    base = {"arm": arm, "model": model, "landmark": landmark, "config": config_dir, "feature_subset": feature_subset, "endpoint": "irae"}
    if not metrics_path.exists():
        rows.append({**base, "n_test": None, "n_test_events": None, "c_index": None,
                     "mean_auc_t": None, "integrated_brier": None, "status": "missing"})
        continue
    df = pd.read_csv(metrics_path)
    irae = df.loc[df["endpoint"] == "irae"]
    if irae.empty:
        rows.append({**base, "n_test": None, "n_test_events": None, "c_index": None,
                     "mean_auc_t": None, "integrated_brier": None, "status": "no irae row"})
        continue
    irae = irae.iloc[0]
    rows.append({
        **base,
        "n_test": int(irae[cols["n_test"]]),
        "n_test_events": int(irae[cols["n_test_events"]]),
        "c_index": float(irae[cols["c_index"]]),
        "mean_auc_t": float(irae[cols["mean_auc_t"]]),
        "integrated_brier": float(irae[cols["integrated_brier"]]),
        "status": "ok",
    })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["arm", "landmark", "model", "config"]).reset_index(drop=True)
summary_df


In [ ]:
# Univariate writes a per-feature association table (cox_agg_univariate_
# nobs_adjusted.csv), not a single metrics row -- summarize hit counts instead.
univariate_rows = []
for arm, model, landmark, config_dir, metrics_filename, feature_subset in TASKS:
    if model != "univariate":
        continue
    row_output_dir = task_output_dir(arm, model, landmark, config_dir)
    uni_path = row_output_dir / metrics_filename
    if not uni_path.exists():
        univariate_rows.append({"arm": arm, "landmark": landmark, "config": config_dir, "feature_subset": feature_subset, "status": "missing"})
        continue
    uni = pd.read_csv(uni_path)
    uni = uni.loc[uni["endpoint"] == "irae"]
    top = uni.sort_values("p_value").iloc[0] if not uni.empty else None
    univariate_rows.append({
        "arm": arm,
        "landmark": landmark,
        "config": config_dir,
        "feature_subset": feature_subset,
        "n_features_tested": len(uni),
        "n_sig_p_lt_05": int((uni["p_value"] < 0.05).sum()) if not uni.empty else 0,
        "n_sig_q_lt_05": int((uni["q_value"] < 0.05).sum()) if not uni.empty else 0,
        "top_feature": top["feature"] if top is not None else None,
        "top_p_value": float(top["p_value"]) if top is not None else None,
        "status": "ok" if not uni.empty else "no irae rows",
    })

univariate_summary_df = pd.DataFrame(univariate_rows).sort_values(["arm", "landmark"]).reset_index(drop=True)
univariate_summary_df
